Missing Data Analysis and Imputation Techniques in Agricultural Dataset Using Pandas



In [ ]:
import pandas as pd
import numpy as np

# Agricultural data with missing values
data = {
    'Farm_ID': ['F001', 'F002', 'F003', 'F004', 'F005', 'F006'],
    'Crop': ['Wheat', 'Rice', np.nan, 'Wheat', 'Corn', 'Rice'],
    'Yield_tons': [180, np.nan, 416, 164, np.nan, 228],
    'Rainfall_mm': [450, 820, np.nan, 420, 710, 850],
    'Fertilizer_kg': [120, 150, 200, np.nan, 195, 160],
    'Soil_pH': [6.5, 5.8, 6.2, 6.7, np.nan, 5.9]
}

df = pd.DataFrame(data)

print("ORIGINAL DATA WITH MISSING VALUES:")
print(df)

print(("\n === MISSING VALUE ANALYSIS === "))
print("\n Missing Values Per Column :")
print(df.isna().sum())
print(f"\n Total Missing Values : {df.isna().sum().sum()}")
print(f"Percentage of Missing Data : {(df.isna().sum().sum()/ df.size*100):.2f}%")

print("\n === ROWS WITH MISSING DATA === ")
rows_with_missing=df[df.isna().any(axis=1)]
print(rows_with_missing)

# Stratergy 1 : Drops rows with missing values
print("\n === STRATERGY 1 : DROP MISSING ROWS === ")
df_dropped=df.dropna()
print(f"Original Shape : {df.shape}")
print(f"After Dropping : {df_dropped.shape}")
print(df_dropped)

# Stratergy 2 : Fill with mean/median
print("\n === STRATERGY 2 : FILL WITH STATISTICS === ")
df_filled=df.copy()
# Fill numeric columns with mean
df_filled["Yield_tons"] = df_filled["Yield_tons"].fillna(df_filled["Yield_tons"].mean())
df_filled["Rainfall_mm"] = df_filled["Rainfall_mm"].fillna(df_filled["Rainfall_mm"].median())
df_filled["Fertilizer_kg"] = df_filled["Fertilizer_kg"].fillna(df_filled["Fertilizer_kg"].mean())
df_filled["Soil_pH"] = df_filled["Soil_pH"].fillna(df_filled["Soil_pH"].median())
# Fill categorical with mode
df_filled["Crop"] = df_filled["Crop"].fillna(df_filled["Crop"].mode()[0]) # .mode() returns a Series, take the first value

# Stratergy 3 : Forward Fill (useful for time series)
print("\n === Startergy 3 : FORWARD FILL ===")
df_ffill=df.ffill()
print(df_ffill)

# Stratergy 4 : Custom fill per column
print("\n === STRATERGY 4 : CUSTOM FILL === ")
df_custom=df.fillna({
    'Crop': 'Unknown',
    'Yield_tons': df['Yield_tons'].median(),
    'Rainfall_mm': 650,  # Regional average
    'Fertilizer_kg': 150,  # Standard recommendation
    'Soil_pH': 6.3  # Neutral pH
})
print(df_custom)

print("\n=== VERIFICATION===")
print(f"Missing values after treatment: {df_custom.isna().sum().sum()}")

ORIGINAL DATA WITH MISSING VALUES:
  Farm_ID   Crop  Yield_tons  Rainfall_mm  Fertilizer_kg  Soil_pH
0    F001  Wheat       180.0        450.0          120.0      6.5
1    F002   Rice         NaN        820.0          150.0      5.8
2    F003    NaN       416.0          NaN          200.0      6.2
3    F004  Wheat       164.0        420.0            NaN      6.7
4    F005   Corn         NaN        710.0          195.0      NaN
5    F006   Rice       228.0        850.0          160.0      5.9

 === MISSING VALUE ANALYSIS === 

 Missing Values Per Column :
Farm_ID          0
Crop             1
Yield_tons       2
Rainfall_mm      1
Fertilizer_kg    1
Soil_pH          1
dtype: int64

 Total Missing Values : 6
Percentage of Missing Data : 16.67%

 === ROWS WITH MISSING DATA === 
  Farm_ID   Crop  Yield_tons  Rainfall_mm  Fertilizer_kg  Soil_pH
1    F002   Rice         NaN        820.0          150.0      5.8
2    F003    NaN       416.0          NaN          200.0      6.2
3    F004  Wheat 

Data Aggregation and Grouping

In [ ]:
import pandas as pd

# Multi-year farm data
data = {
    'Year': [2020, 2020, 2020, 2021, 2021, 2021, 2022, 2022, 2022, 2023, 2023, 2023],
    'Region': ['North', 'South', 'East', 'North', 'South', 'East',
               'North', 'South', 'East', 'North', 'South', 'East'],
    'Crop': ['Wheat', 'Rice', 'Corn', 'Wheat', 'Rice', 'Corn',
             'Wheat', 'Rice', 'Corn', 'Wheat', 'Rice', 'Corn'],
    'Yield_tons': [180, 228, 416, 195, 245, 430, 210, 250, 445, 225, 260, 455],
    'Area_ha': [50, 45, 55, 52, 46, 56, 54, 48, 57, 55, 49, 58],
    'Cost_thousands': [85, 95, 180, 90, 100, 190, 95, 105, 195, 100, 110, 200]
}

df = pd.DataFrame(data)

print('AGRICULTURAL DATASET :')
print(df)

# Group by Region
print('\n=== ANALYSIS BY REGION ===')
region_stats = df.groupby('Region').agg({
    'Yield_tons': ['sum', 'mean', 'std'],
    'Area_ha': 'sum',
    'Cost_thousands': 'sum'
})
print(region_stats)

# Group by Crop
print('\n=== ANALYSIS BY CROP ===')
crop_stats = df.groupby('Crop').agg({
    'Yield_tons': ['mean', 'min', 'max'],
    'Cost_thousands': 'mean'
}).round(2)
print(crop_stats)

# Multiple grouping (Region and Crop)
print("\n === REGION-CROP ANALYSIS === ")
region_crop=df.groupby(["Region","Crop"])["Yield_tons"].mean().round(2)
print(region_crop)

# Year-over-year growth
print("\n === YEARLY TRENDS === ")
yearly=df.groupby("Year").agg({
    'Yield_tons': 'sum',
    'Area_ha': 'sum'
})
yearly['Yield_per_ha'] = (yearly['Yield_tons'] / yearly['Area_ha']).round(2)
print(yearly)


# Transform : Add group mean to each row
df['Region_Avg_Yield'] = df.groupby('Region')['Yield_tons'].transform('mean').round(2)
df['Crop_Avg_Yield'] = df.groupby('Crop')['Yield_tons'].transform('mean').round(2)

print('\n=== DATASET WITH GROUP AVERAGES ===')
print(df[['Year', 'Region', 'Crop', 'Yield_tons', 'Region_Avg_Yield', 'Crop_Avg_Yield']].head(6))

# Pivot table
print('\n=== PIVOT TABLE: Average Yield by Region and Crop ===')
pivot = df.pivot_table(values='Yield_tons',
                       index='Region',
                       columns='Crop',
                       aggfunc='mean').round(2)
print(pivot)

# Filter groups with high performance
print('\n=== HIGH PERFORMING REGIONS (Avg Yield > 300) ===')
high_regions = df.groupby('Region').filter(lambda x: x['Yield_tons'].mean() > 300)
print(high_regions[['Region', 'Crop', 'Yield_tons']].drop_duplicates('Region'))





AGRICULTURAL DATASET :
    Year Region   Crop  Yield_tons  Area_ha  Cost_thousands
0   2020  North  Wheat         180       50              85
1   2020  South   Rice         228       45              95
2   2020   East   Corn         416       55             180
3   2021  North  Wheat         195       52              90
4   2021  South   Rice         245       46             100
5   2021   East   Corn         430       56             190
6   2022  North  Wheat         210       54              95
7   2022  South   Rice         250       48             105
8   2022   East   Corn         445       57             195
9   2023  North  Wheat         225       55             100
10  2023  South   Rice         260       49             110
11  2023   East   Corn         455       58             200

=== ANALYSIS BY REGION ===
       Yield_tons                    Area_ha Cost_thousands
              sum    mean        std     sum            sum
Region                                           